In [ ]:
import yaml
import torch
from pathlib import Path
from datasets import load_from_disk
from diffusers import StableDiffusionPipeline, DDPMScheduler
from diffusers.loaders import AttnProcsLayers
from diffusers.models.attention_processor import LoRAAttnProcessor
from transformers import CLIPTokenizer, CLIPTextModel
from torch.utils.data import DataLoader
from torchvision import transforms
from torch.optim import AdamW

In [ ]:
with open("/Users/hannafagrell/Documents/8.SEMESTER/INF-3600/SwatchMagic/training/training_config.yaml") as f:
    config = yaml.safe_load(f)

MODEL_ID       = config["model_id"]
DATASET_PATH   = config["dataset_path"]
OUTPUT_DIR     = Path(config["output_dir"])
LEARNING_RATE  = config["learning_rate"]
NUM_EPOCHS     = config["num_epochs"]
BATCH_SIZE     = config["batch_size"]
LORA_RANK      = config["lora_rank"]
IMAGE_SIZE     = config["image_size"]
MIXED_PRECISION = config["mixed_precision"]  # "fp16" or "no"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
print("Loading model...")
tokenizer   = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder").to(DEVICE)
pipeline    = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16 if MIXED_PRECISION == "fp16" else torch.float32)
unet        = pipeline.unet.to(DEVICE)
vae         = pipeline.vae.to(DEVICE)
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

In [ ]:
# Freeze everything except LoRA layers
vae.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

In [ ]:
print(f"Injecting LoRA adapters (rank={LORA_RANK})...")
lora_attn_procs = {}
for name in unet.attn_processors.keys():
    cross_attention_dim = None if name.endswith("attn1.processor") else unet.config.cross_attention_dim
    hidden_size = unet.config.block_out_channels[0]
    lora_attn_procs[name] = LoRAAttnProcessor(
        hidden_size=hidden_size,
        cross_attention_dim=cross_attention_dim,
        rank=LORA_RANK
    )

unet.set_attn_processor(lora_attn_procs)
lora_layers = AttnProcsLayers(unet.attn_processors)


In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

def preprocess(example):
    example["pixel_values"] = transform(example["image"].convert("RGB"))
    tokens = tokenizer(example["text"], padding="max_length", truncation=True, max_length=77, return_tensors="pt")
    example["input_ids"] = tokens.input_ids[0]
    return example

print("Loading dataset...")
dataset = load_from_disk(DATASET_PATH)
dataset = dataset.map(preprocess, remove_columns=["image", "text"])
dataset.set_format("torch")
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
optimizer = AdamW(lora_layers.parameters(), lr=LEARNING_RATE)

In [ ]:
print("Starting training...")
for epoch in range(NUM_EPOCHS):
    total_loss = 0
    for step, batch in enumerate(dataloader):
        pixel_values = batch["pixel_values"].to(DEVICE)
        input_ids    = batch["input_ids"].to(DEVICE)

        # Encode images to latent space
        with torch.no_grad():
            latents = vae.encode(pixel_values).latent_dist.sample() * 0.18215

        # Sample noise and timesteps
        noise      = torch.randn_like(latents)
        timesteps  = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device=device).long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        # Encode text
        with torch.no_grad():
            encoder_hidden_states = text_encoder(input_ids)[0]

        # Predict noise and compute loss
        noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
        loss = torch.nn.functional.mse_loss(noise_pred, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} — loss: {avg_loss:.4f}")


In [ ]:
# plot loss curve

In [ ]:
print("Saving LoRA weights...")
unet.save_attn_procs(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")